# 🌸 鳶尾花隨機森林分類器 Gradio Web UI 實作教學（分段講解與程式碼複製貼上）

> 對象：修習機器學習模型部署與 Web UI 開發的學員
> 對應程式碼：`app.py` 第 171~476 行 (`Gradio UI` 區塊)
>
> 本 Notebook 採用**分段拆解教學法**：每一區塊先講解核心觀念，再提供可以直接複製貼入 `app.py` 的程式碼，並詳細說明「為什麼要這樣寫」。

## 🎯 學習目標

- [ ] 能使用 HTML/CSS 打造現代化的視覺卡片（預測品種卡、機率動態橫條圖、指標卡、特徵重要性圖表）
- [ ] 理解 Gradio 事件處理器（Handler）如何接收前端組件輸入並回傳動態 HTML
- [ ] 掌握 Gradio `Blocks` 網頁排版架構（`Tabs`, `Tab`, `Row`, `Column` 雙欄響應式設計）
- [ ] 了解滑桿拖曳時 `.release()` 與 `.change()` 事件對伺服器效能與 Render 反向代理的影響
- [ ] 掌握 Gradio 主題設定與在生產環境（FastAPI + Uvicorn）防止 500 錯誤的 `theme_hash` 設定

---
## 📌 步驟 1：HTML 視覺卡片生成函數 (UI Component Builders)

### 💡 核心觀念
Gradio 內建的 `gr.HTML` 允許我們寫原生 HTML/CSS。比起標準文字，包裝成 **現代化儀表板卡片**（顏色區分 Setosa/Versicolor/Virginica、直覺進度條）能極大地提升 UX。

### 📋 測試 HTML 生成函數：

In [ ]:
def make_prediction_card(label: str, prob: float) -> str:
    color_map = {
        "setosa": ("#e6f4ea", "#137333", "🌿 Setosa (山鳶尾)"),
        "versicolor": ("#fef7e0", "#b06000", "🍁 Versicolor (變色鳶尾)"),
        "virginica": ("#fce8e6", "#c5221f", "🪻 Virginica (維吉尼亞鳶尾)")
    }
    bg, fg, name = color_map.get(label, ("#f8f9fa", "#212529", label))
    return f"""
    <div style="background-color: {bg}; color: {fg}; padding: 22px; border-radius: 12px; border: 1.5px solid {fg}40; text-align: center; margin-bottom: 20px; box-shadow: 0 4px 12px rgba(0,0,0,0.06);">
        <span style="font-size: 0.95rem; font-weight: bold; text-transform: uppercase; letter-spacing: 1.5px; opacity: 0.85;">預測分析品種</span>
        <h2 style="font-size: 2.4rem; margin: 8px 0; font-weight: 800;">{name}</h2>
        <span style="font-size: 1.1rem; font-weight: 500;">預測機率: <strong style="font-size: 1.5rem;">{prob:.1f}%</strong></span>
    </div>
    """

def make_probability_bars(prob_dict: dict[str, float]) -> str:
    color_scheme = {
        "setosa": "#137333",
        "versicolor": "#b06000",
        "virginica": "#c5221f"
    }
    html = '<div style="margin-top: 10px; display: flex; flex-direction: column; gap: 14px;">'
    for cls, val in prob_dict.items():
        pct = val * 100
        color = color_scheme.get(cls, "#0dcaf0")
        html += f"""
        <div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 5px; font-weight: 600;">
                <span style="text-transform: capitalize;">{cls}</span>
                <span>{pct:.1f}%</span>
            </div>
            <div style="background-color: #f1f3f4; border-radius: 8px; height: 12px; overflow: hidden; width: 100%;">
                <div style="background-color: {color}; width: {pct}%; height: 100%; border-radius: 8px; transition: width 0.6s ease;"></div>
            </div>
        </div>
        """
    html += '</div>'
    return html

# 測試預覽 HTML 效果
sample_card = make_prediction_card('setosa', 98.5)
sample_bars = make_probability_bars({'setosa': 0.985, 'versicolor': 0.012, 'virginica': 0.003})
print("預測卡片 HTML 產生成功！長度：", len(sample_card))
print("機率條 HTML 產生成功！長度：", len(sample_bars))

---
## 📌 步驟 2：Gradio 事件處理器 (Event Handlers)

編寫 `predict_gradio_handler` 與 `train_gradio_handler` 函數接管 UI 按鈕與滑桿事件。

In [ ]:
import os, sys, joblib
current_dir = os.getcwd()
model_path = os.path.join(current_dir, 'iris_model.joblib')
if not os.path.exists(model_path):
    from train_save import train_and_save_model
    train_and_save_model()

model_data = joblib.load(model_path)

def predict_gradio_handler(sepal_len, sepal_wid, petal_len, petal_wid):
    features = [[sepal_len, sepal_wid, petal_len, petal_wid]]
    model = model_data["model"]
    target_names = model_data["target_names"]
    pred_id = int(model.predict(features)[0])
    pred_label = target_names[pred_id]
    probs = model.predict_proba(features)[0]
    prob_dict = {target_names[i]: float(p) for i, p in enumerate(probs)}
    card_html = make_prediction_card(pred_label, prob_dict[pred_label] * 100)
    bars_html = make_probability_bars(prob_dict)
    return card_html, bars_html

# 測試事件處理器
card, bars = predict_gradio_handler(5.1, 3.5, 1.4, 0.2)
print("事件處理器執行成功！")

---
## 📌 步驟 3：建構 Gradio Blocks 頁面佈局與效能優化

重點說明：
- 採用 `gr.Blocks()` 與 `gr.Tabs()` 響應式雙欄佈局
- **`slider.release(...)` 代替 `.change()`**：放開滑桿才發送請求，防止拖曳時數十個請求卡爆 CPU
- **`queue=False` 與 `show_progress='hidden'`**：避開 Render 反向代理對 SSE 長連線的緩衝問題，畫面平順流暢不卡頓

In [ ]:
import gradio as gr

with gr.Blocks(title="🌸 Iris 鳶尾花分類器") as demo:
    gr.Markdown("# 🌸 Iris 鳶尾花機器學習全生命週期平台")
    with gr.Tabs():
        with gr.Tab("🔮 即時模型預測"):
            with gr.Row():
                with gr.Column():
                    s_len = gr.Slider(0.1, 10.0, value=5.1, label="花萼長度 Sepal Length (cm)")
                    s_wid = gr.Slider(0.1, 10.0, value=3.5, label="花萼寬度 Sepal Width (cm)")
                    p_len = gr.Slider(0.1, 10.0, value=1.4, label="花瓣長度 Petal Length (cm)")
                    p_wid = gr.Slider(0.1, 10.0, value=0.2, label="花瓣寬度 Petal Width (cm)")
                    btn = gr.Button("🔮 開始預測", variant="primary")
                with gr.Column():
                    out_card = gr.HTML()
                    out_bars = gr.HTML()
            
            inputs = [s_len, s_wid, p_len, p_wid]
            outputs = [out_card, out_bars]
            for slider in inputs:
                slider.release(predict_gradio_handler, inputs, outputs, queue=False, show_progress="hidden")
            btn.click(predict_gradio_handler, inputs, outputs, queue=False, show_progress="hidden")

print("Gradio Blocks 佈局建構完成！")

---
## 📌 步驟 4：FastAPI / Uvicorn 掛載與 Theme Hash 修正

當不使用 `demo.launch()` 而是直接由 Uvicorn 載入 `app` 時，必須手動計算主題 Hash 以防 500 錯誤：

In [ ]:
import hashlib

demo.theme = gr.themes.Soft(primary_hue="teal", secondary_hue="indigo")
demo.theme_css = demo.theme._get_theme_css()
demo.stylesheets = demo.theme._stylesheets
demo.theme_hash = hashlib.sha256(demo.theme_css.encode('utf-8')).hexdigest()

# 掛載至 FastAPI
app = gr.routes.App.create_app(demo)
print(f"Theme Hash 計算完成 ({demo.theme_hash[:8]}...)，Gradio App 成功建立並可隨時掛載進 FastAPI！")